# 03 – Phân tích lỗi (Task 11 Step 7–8)
Logic nằm trong `absa.errors`. Cần file dự đoán trong `results/predictions/` (không commit; tải từ Drive `nlp-absa-visfd/backups/<thời điểm>/results/predictions/` về cùng chỗ).

Chỉ phân tích lỗi trên **Dev**. Confusion matrix trên Test ở cuối chỉ để báo cáo, không dùng để chọn gì.

In [ ]:
from pathlib import Path

import pandas as pd

from absa.errors import (
    ERROR_TYPES,
    collect_errors,
    load_predictions,
    plot_confusions,
    sample_errors,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PRED, OUT = ROOT / "results" / "predictions", ROOT / "results" / "error_analysis"
TAB, FIG = ROOT / "results" / "tables", ROOT / "results" / "figures"
OUT.mkdir(parents=True, exist_ok=True)

# Mô hình transformer tốt nhất và mốc TF-IDF (tên file dự đoán, không gồm __dev.csv)
MODEL = "final_phobert__seed42"
BASELINE = "final_tfidf__seed42"

## 1. Thu thập lỗi trên Dev

In [ ]:
df, Y_true, Y_pred = load_predictions(PRED / f"{MODEL}__dev.csv")
errors = collect_errors(df, Y_true, Y_pred)
errors.to_csv(OUT / "dev_errors_final_phobert.csv", index=False)
print(f"{len(errors)}/{len(df)} bình luận có ít nhất một lỗi")
errors[["missed", "spurious", "wrong_polarity"]].map(len).sum()

## 2. Lấy mẫu 40 lỗi để gán nhãn tay
Ô này chỉ ghi file khi file **chưa có**, để không xóa nhãn đã điền. Mở `results/error_analysis/dev_errors_sample_labeled.csv` (Excel hoặc VS Code), điền:
- `error_type`: một giá trị trong `ERROR_TYPES`;
- `cause`: nguyên nhân giả định; `fix`: hướng khắc phục;
- `selected`: ghi `x` cho 8–10 ví dụ đưa vào báo cáo (phủ ít nhất 5 loại lỗi).

Lưu lại dạng CSV UTF-8 rồi chạy tiếp từ mục 3.

In [ ]:
LABELED = OUT / "dev_errors_sample_labeled.csv"
if not LABELED.exists():
    sample_errors(errors, n=40, seed=42).assign(selected="").to_csv(
        LABELED, index=False, encoding="utf-8-sig"
    )
print(ERROR_TYPES)

## 3. Thống kê loại lỗi

In [ ]:
labeled = pd.read_csv(LABELED, keep_default_na=False, encoding="utf-8-sig")
bad = labeled[~labeled["error_type"].isin(ERROR_TYPES)]
assert bad.empty, f"error_type chưa điền hoặc sai ở index {bad['index'].tolist()}"
labeled["error_type"].value_counts()

## 4. So sánh chéo TF-IDF và PhoBERT trên Dev

In [ ]:
base_df, B_true, B_pred = load_predictions(PRED / f"{BASELINE}__dev.csv")
assert base_df["index"].tolist() == df["index"].tolist()
model_ok = (Y_pred == Y_true).all(axis=1)
base_ok = (B_pred == B_true).all(axis=1)
cross = pd.DataFrame(
    {
        "case": [
            "both_correct",
            "tfidf_wrong_phobert_correct",
            "tfidf_correct_phobert_wrong",
            "both_wrong",
        ],
        "n_comments": [
            int((base_ok & model_ok).sum()),
            int((~base_ok & model_ok).sum()),
            int((base_ok & ~model_ok).sum()),
            int((~base_ok & ~model_ok).sum()),
        ],
    }
)
cross.to_csv(TAB / "cross_errors_dev.csv", index=False)
cross

## 5. Ví dụ tiêu biểu cho báo cáo

In [ ]:
chosen = labeled[labeled["selected"].str.strip() != ""]
assert 8 <= len(chosen) <= 10, f"cần chọn 8–10 ví dụ, đang có {len(chosen)}"
assert chosen["error_type"].nunique() >= 5, "cần phủ ít nhất 5 loại lỗi"
cols = {
    "comment": "Bình luận",
    "n_star": "Sao",
    "gold": "Nhãn đúng",
    "pred": "Dự đoán",
    "error_type": "Loại lỗi",
    "cause": "Nguyên nhân giả định",
    "fix": "Hướng khắc phục",
}
table = chosen[list(cols)].rename(columns=cols)
table["Bình luận"] = table["Bình luận"].str.replace("\n", " ").str.replace("|", "/")
(OUT / "selected_examples.md").write_text(
    "# Ví dụ lỗi tiêu biểu (Dev)\n\n" + table.to_markdown(index=False) + "\n",
    encoding="utf-8",
)
table

## 6. Confusion matrix trên Test (Step 8)
Chỉ chạy sau khi đã đánh giá Test (Step 5). Seed 42, chỉ để báo cáo.

In [ ]:
test_df, T_true, T_pred = load_predictions(PRED / f"{MODEL}__test.csv")
plot_confusions(
    T_true,
    T_pred,
    ["GENERAL", "PERFORMANCE", "BATTERY", "FEATURES", "STORAGE"],
    FIG / "confusion_test_final_phobert.png",
)